In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, datetime, timedelta
# web scrapping
import bs4 as bs
import requests
import lxml
from functools import reduce
# matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from ipysigma import Sigma
from pyvis.network import Network
import requests
import re
from bs4 import BeautifulSoup
from io import StringIO
from dbconnection import MySQLDatabase
from utils import getSymbols, getData, get_last_date, get_marketid_simbols
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="pandas")
sns.set_theme()

In [2]:
db = MySQLDatabase("financialmarkets")

In [3]:
def getSymbols(url):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    resp = requests.get(url, headers=headers)

    if resp.status_code != 200:
        raise ValueError(f"Error al obtener la página: {resp.status_code}")

    soup = BeautifulSoup(resp.text, "lxml")

    # Buscar cualquier tabla con clase 'wikitable'
    table = soup.find("table", {"id": lambda x: x and "constituents" in x})
    #print(len(table[]))
    #print(table)
    if table is None:
        raise ValueError("No se encontró ninguna tabla con clase 'wikitable'")

    # Intentar identificar la tabla correcta: debe contener columna con "Company" o "Firma"
    
    # Convertir tabla a DataFrame
    table = pd.read_html(StringIO(str(table)))[0]

    return table

In [15]:
df = getSymbols('https://en.wikipedia.org/wiki/FTSE_100_Index')
df.head()

,Company,Ticker,FTSE industry classification benchmark sector[37]
0,3i,III,Financial services
1,Admiral Group,ADM,Insurance
2,Airtel Africa,AAF,Telecommunications services
3,Alliance Witan,ALW,Investment Trusts
4,Anglo American plc,AAL,Mining


In [16]:
# generamos variables no existentes
df['Ticker'] = [re.sub(r"\s+", "", x)+'.L' for x in df['Ticker'].astype('str')]
df = df.rename(columns={'Ticker':'Symbol','Company':'Security', 'FTSE industry classification benchmark sector[37]':'GICS Sector'})
df['GICS Sub-Industry'] = ['SD' for x in df['Symbol']]
df['Headquarters Location'] = ['SD' for x in df['Symbol']]
df['CIK'] = ['SD' for x in df['Symbol']]
df['Founded'] = [9999 for x in df['Symbol']]
df['Date added'] = ['0000-00-00' for x in df['Symbol']]
df = df[['Symbol','Security','GICS Sector','GICS Sub-Industry','Headquarters Location', 'Date added','CIK','Founded']]
df.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,III.L,3i,Financial services,SD,SD,0000-00-00,SD,9999
1,ADM.L,Admiral Group,Insurance,SD,SD,0000-00-00,SD,9999
2,AAF.L,Airtel Africa,Telecommunications services,SD,SD,0000-00-00,SD,9999
3,ALW.L,Alliance Witan,Investment Trusts,SD,SD,0000-00-00,SD,9999
4,AAL.L,Anglo American plc,Mining,SD,SD,0000-00-00,SD,9999


**Ingresamos mercado**

In [17]:
# ---------------------------
# 3️Insertar/actualizar mercados
# ---------------------------
markets = pd.DataFrame({
    'market_name': ['FTSE100'],
    'country': ['UKX'],
    'currency': ['GBP']
})
markets

,market_name,country,currency
0,FTSE100,UKX,GBP


In [18]:
db.insert_to_db(markets, tabla="markets", batch_size=5000)

✅ Conexión exitosa


In [19]:
# Obtener market_id
market_id = db.execute_query("SELECT * FROM markets")
market_id

,market_id,market_name,country,currency
0,1,NASDAQ,USA,USD
1,2,S&P 500,USA,USD
2,3,IPC MX,MX,Peso
3,4,DAX,GERMAN,DEM
4,5,FTSE100,UKX,GBP


In [20]:
market_id = 5

In [21]:
df.loc[:,"market_id"] = [market_id for x in df['Symbol']]
companies = df[["market_id",'Symbol','Security','GICS Sector','GICS Sub-Industry','Date added','Headquarters Location','CIK','Founded']]
companies.columns = ["market_id",'symbol','name','sector_name','sub_industry','date_added','headquarters','cik','founded']
companies = companies.reset_index(drop=True)
companies.head()

,market_id,symbol,name,sector_name,sub_industry,date_added,headquarters,cik,founded
0,5,III.L,3i,Financial services,SD,0000-00-00,SD,SD,9999
1,5,ADM.L,Admiral Group,Insurance,SD,0000-00-00,SD,SD,9999
2,5,AAF.L,Airtel Africa,Telecommunications services,SD,0000-00-00,SD,SD,9999
3,5,ALW.L,Alliance Witan,Investment Trusts,SD,0000-00-00,SD,SD,9999
4,5,AAL.L,Anglo American plc,Mining,SD,0000-00-00,SD,SD,9999


In [22]:
db.insert_to_db(companies, tabla="companies", batch_size=100)

In [23]:
db.close()

🔒 Conexión cerrada
